In [ ]:
from datasets import load_dataset

# Problem bei normalem Laden
dataset = load_dataset(
    "ptb_text_only",
    split="train",
    trust_remote_code=True
)

ds_split = dataset.train_test_split(test_size=0.2)
train_set = ds_split["train"]
test_set = ds_split["test"]

print(train_set)
print(test_set)
print(train_set[0])
print(test_set[0])

Dataset({
    features: ['sentence'],
    num_rows: 33654
})
Dataset({
    features: ['sentence'],
    num_rows: 8414
})
{'sentence': "we 've approached this not as a response to a temporary condition in the industry but as a fundamental change the industry is going through mr. <unk> said"}
{'sentence': 'the philadelphia fed says it is merely an agent for coins responsible only for storage and distribution'}


In [4]:
# Token <stop> an jeden Satz anhängen
train_set = train_set.map(lambda x: {"sentence": x["sentence"] + " <stop>"})
test_set = test_set.map(lambda x: {"sentence": x["sentence"] + " <stop>"})


# Sätze mit weniger als 3 Wörtern entfernen
train_set = train_set.filter(lambda x: len(x["sentence"].split()) >= 3)
test_set = test_set.filter(lambda x: len(x["sentence"].split()) >= 3)

print(train_set[0])
print(test_set[0])

Map:   0%|          | 0/33654 [00:00<?, ? examples/s]

Map:   0%|          | 0/8414 [00:00<?, ? examples/s]

Filter:   0%|          | 0/33654 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8414 [00:00<?, ? examples/s]

{'sentence': "we 've approached this not as a response to a temporary condition in the industry but as a fundamental change the industry is going through mr. <unk> said <stop>"}
{'sentence': 'the philadelphia fed says it is merely an agent for coins responsible only for storage and distribution <stop>'}


In [ ]:
from collections import defaultdict

def estimate_unigram(train_set):
    # leeres dict erstellen (default = 0)
    unigram_model = defaultdict(int)

    for example in train_set:
        # Satz holen und in Wörter teilen
        sentence = example["sentence"]
        words = sentence.split()
        
        for word in words:
            unigram_model[word] += 1

    return unigram_model

In [9]:
unigram_model = estimate_unigram(train_set)
print(unigram_model["the"])

40776


In [ ]:
import math

def unigram_sentence_logp(sentence, unigram_model):
    words = sentence.split()
    total_count = sum(unigram_model.values())
    logp = 0

    for word in words:
        count = unigram_model[word]

        # unbekanntes Wort = -inf sonst match.log(0) -> error
        if count == 0:
            return float("-inf")
        
        word_prob = count / total_count
        logp += math.log(word_prob)

    return logp

In [24]:
sentence_1 = "the the the <stop>"
sentence_2 = "i love computer science <stop>"

print(unigram_sentence_logp(sentence_1, unigram_model))
print(unigram_sentence_logp(sentence_2, unigram_model))

-11.811779223348623
-38.42204822162591


In [28]:
import numpy as np

def unigram_perplexity(dataset, unigram_model):
    total_logp = 0
    total_words = 0

    for example in dataset:
        # Satz holen und in Wörter teilen
        sentence = example["sentence"]
        words = sentence.split()

        logp = unigram_sentence_logp(sentence, unigram_model)

        # Wenn Satz unendliche Wahrscheinlichkeit hat, überspringen
        if logp == np.inf or logp == -np.inf:
            continue

        total_logp += logp
        total_words += len(words)

    # P(W)^-1/n == exp(-1/n * logP(W))
    perplexity = np.exp(-total_logp / total_words)

    return perplexity

In [ ]:
test_perplexity = unigram_perplexity(test_set, unigram_model)
print(test_perplexity)

691.506071785476


In [ ]:
def remove_rares(dataset1, dataset2, threshold):
    word_counts = estimate_unigram(dataset1)

    # wenn word_count groß genug gleich bleiben sonst <unk>
    # erst Liste in String umwandeln
    dataset1 = dataset1.map(
        lambda x: {"sentence": " ".join([w if word_counts[w] >= threshold else "<unk>" for w in x["sentence"].split()])})

    dataset2 = dataset2.map(
        lambda x: {"sentence": " ".join([w if word_counts[w] >= threshold else "<unk>" for w in x["sentence"].split()])})

    return dataset1, dataset2

In [38]:
clean_train_set, clean_test_set = remove_rares(train_set, test_set, threshold=3)
clean_unigram_model = estimate_unigram(clean_train_set)
test_perplexity = unigram_perplexity(clean_test_set, clean_unigram_model)
print(clean_train_set[0])
print(clean_test_set[0])
print(test_perplexity)

Map:   0%|          | 0/33538 [00:00<?, ? examples/s]

Map:   0%|          | 0/8393 [00:00<?, ? examples/s]

{'sentence': "we 've approached this not as a response to a temporary condition in the industry but as a fundamental change the industry is going through mr. <unk> said <stop>"}
{'sentence': 'the philadelphia fed says it is merely an agent for coins responsible only for storage and distribution <stop>'}
682.7549058993617


In [39]:
from collections import defaultdict

def estimate_bigram(dataset):
    unigram_model = estimate_unigram(dataset)
    bigram_model = defaultdict(int)

    for example in dataset:
        sentence = example["sentence"]
        words = sentence.split()

        # Häufigkeiten von Wortpaaren
        # zip um Paare zu erstellen
        for word1, word2 in zip(words[:-1], words[1:]):
            bigram_model[(word1, word2)] += 1

    return unigram_model, bigram_model

In [47]:
unigram_model, bigram_model = estimate_bigram(clean_train_set)
print(unigram_model["the"])
print(bigram_model[("in", "the")])

40776
3592


In [ ]:
def bigram_sentence_logp(sentence, unigram_model, bigram_model):
    words = sentence.split()
    logp = 0

    for word1, word2 in zip(words[:-1], words[1:]):
        bigram_count = bigram_model[(word1, word2)]
        unigram_count = unigram_model[word1]

        # unbekannte Wörter
        if bigram_count == 0 or unigram_count == 0:
            return float("-inf")

        prob = bigram_count / unigram_count
        logp += math.log(prob)

    return logp

In [49]:
zero_prob_count = 0

for example in clean_test_set:
    sentence = example["sentence"]
    logp = bigram_sentence_logp(sentence, unigram_model, bigram_model)

    if logp == np.inf or logp == -np.inf:
        zero_prob_count += 1

print(zero_prob_count)

7647


In [50]:
sentence_1 = "the the the <stop>"
sentence_2 = "i love computer science <stop>"

print(bigram_sentence_logp(sentence_1, unigram_model, bigram_model))
print(bigram_sentence_logp(sentence_2, unigram_model, bigram_model))

-26.184586375923686
-inf


In [ ]:
def estimate_bigram_smoothed(dataset, a):
    unigram_model, bigram_model = estimate_bigram(dataset)
    vocab_size = len(unigram_model)

    return unigram_model, bigram_model, a, vocab_size

# Sinn??

In [74]:
def bigram_sentence_logp_smoothed(sentence, unigram_model, bigram_model, a):
    vocab_size = len(unigram_model)
    words = sentence.split()
    logp = 0

    for word1, word2 in zip(words[:-1], words[1:]):
        bigram_count = bigram_model[(word1, word2)]
        unigram_count = unigram_model[word1]

        prob = (bigram_count + a) / (unigram_count + a * vocab_size)
        logp += math.log(prob)

    return logp

In [75]:
sentence_1 = "the the the <stop>"
sentence_2 = "i love computer science <stop>"

print(bigram_sentence_logp_smoothed(sentence_1, unigram_model, bigram_model, 1))
print(bigram_sentence_logp_smoothed(sentence_2, unigram_model, bigram_model, 1))

-26.409749329518743
-35.13845348535458


In [82]:
def bigram_perplexity_smoothed(dataset, unigram_model, bigram_model, a):
    vocab_size = len(unigram_model)
    total_logp = 0
    total_words = 0

    for example in dataset:
        sentence = example["sentence"]
        words = sentence.split()

        logp = bigram_sentence_logp_smoothed(sentence, unigram_model, bigram_model, a)

        total_logp += logp
        total_words += len(words)

    # P(W)^-1/n == exp(-1/n * logP(W))
    return np.exp(-total_logp / total_words)

In [87]:
test_perplexity = bigram_perplexity_smoothed(clean_test_set, unigram_model, bigram_model, 1)
print(test_perplexity)

713.8399740455621
